# Silver Layer - Data Cleaning, Integration and Enrichment

## Objective

The Silver Layer transforms raw Bronze data into a clean, validated and business-ready dataset.

## Silver Layer Responsibilities

- Read Bronze datasets
- Handle null and invalid records
- Remove duplicate records
- Validate IDs and timestamps
- Join drivers, trips and trip logs
- Create derived columns
- Separate valid and rejected records
- Store cleaned data in Parquet format

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
bronze_base_path = "/Volumes/workspace/default/rideshare_data/bronze"

drivers_df = spark.read.parquet(f"{bronze_base_path}/drivers")
trips_df = spark.read.parquet(f"{bronze_base_path}/trips")
trip_logs_df = spark.read.parquet(f"{bronze_base_path}/trip_logs")

print("Drivers:", drivers_df.count())
print("Trips:", trips_df.count())
print("Trip Logs:", trip_logs_df.count())

Drivers: 150
Trips: 150
Trip Logs: 150


In [0]:
print("Duplicate drivers:",
      drivers_df.count() - drivers_df.dropDuplicates(["driver_id"]).count())

print("Duplicate trips:",
      trips_df.count() - trips_df.dropDuplicates(["trip_id"]).count())

print("Duplicate trip logs:",
      trip_logs_df.count() - trip_logs_df.dropDuplicates(["log_id"]).count())

Duplicate drivers: 0
Duplicate trips: 0
Duplicate trip logs: 0


In [0]:
def null_count(df):
    return df.select([
        F.sum(F.col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])

print("Drivers null values")
display(null_count(drivers_df))

print("Trips null values")
display(null_count(trips_df))

print("Trip logs null values")
display(null_count(trip_logs_df))

Drivers null values


driver_id,name,city,rating
0,0,0,0


Trips null values


trip_id,driver_id,pickup_location,drop_location,distance_km,fare_amount,trip_status
0,0,0,0,0,0,0


Trip logs null values


log_id,trip_id,start_time,end_time,delay_minutes,cancellation_flag
0,0,0,87,0,0


In [0]:
trip_logs_df.groupBy("cancellation_flag").agg(
    F.count("*").alias("total_records"),
    F.sum(F.col("end_time").isNull().cast("int")).alias("null_end_time")
).show()

+-----------------+-------------+-------------+
|cancellation_flag|total_records|null_end_time|
+-----------------+-------------+-------------+
|                1|           87|           87|
|                0|           63|            0|
+-----------------+-------------+-------------+



In [0]:
print(
    "Invalid driver ratings:",
    drivers_df.filter(
        (F.col("rating") < 0) | (F.col("rating") > 5)
    ).count()
)

print(
    "Negative trip distances:",
    trips_df.filter(F.col("distance_km") < 0).count()
)

print(
    "Negative fare amounts:",
    trips_df.filter(F.col("fare_amount") < 0).count()
)

print(
    "Negative delays:",
    trip_logs_df.filter(F.col("delay_minutes") < 0).count()
)

Invalid driver ratings: 0
Negative trip distances: 0
Negative fare amounts: 0
Negative delays: 0


In [0]:
status_flag_check = trips_df.join(
    trip_logs_df,
    on="trip_id",
    how="inner"
).filter(
    (
        (F.col("trip_status") == "Completed") &
        (F.col("cancellation_flag") != 0)
    )
    |
    (
        (F.col("trip_status") == "Cancelled") &
        (F.col("cancellation_flag") != 1)
    )
)

print(
    "Trip status and cancellation flag mismatches:",
    status_flag_check.count()
)

Trip status and cancellation flag mismatches: 0


In [0]:
clean_drivers_df = (
    drivers_df
    .dropDuplicates(["driver_id"])
    .filter(F.col("driver_id").isNotNull())
    .filter(F.col("name").isNotNull())
    .filter(F.col("city").isNotNull())
    .filter(F.col("rating").between(0, 5))
)

clean_trips_df = (
    trips_df
    .dropDuplicates(["trip_id"])
    .filter(F.col("trip_id").isNotNull())
    .filter(F.col("driver_id").isNotNull())
    .filter(F.col("distance_km") >= 0)
    .filter(F.col("fare_amount") >= 0)
    .filter(F.col("trip_status").isin("Completed", "Cancelled"))
)

clean_trip_logs_df = (
    trip_logs_df
    .dropDuplicates(["log_id"])
    .filter(F.col("log_id").isNotNull())
    .filter(F.col("trip_id").isNotNull())
    .filter(F.col("start_time").isNotNull())
    .filter(F.col("delay_minutes") >= 0)
    .filter(F.col("cancellation_flag").isin(0, 1))
    .filter(
        (F.col("cancellation_flag") == 1) |
        F.col("end_time").isNotNull()
    )
)

print("Clean drivers:", clean_drivers_df.count())
print("Clean trips:", clean_trips_df.count())
print("Clean trip logs:", clean_trip_logs_df.count())

Clean drivers: 150
Clean trips: 150
Clean trip logs: 150


In [0]:
silver_df = (
    clean_trips_df.alias("t")
    .join(
        clean_drivers_df.alias("d"),
        F.col("t.driver_id") == F.col("d.driver_id"),
        "inner"
    )
    .join(
        clean_trip_logs_df.alias("l"),
        F.col("t.trip_id") == F.col("l.trip_id"),
        "inner"
    )
    .select(
        F.col("t.trip_id"),
        F.col("t.driver_id"),
        F.col("d.name").alias("driver_name"),
        F.col("d.city"),
        F.col("d.rating"),
        F.col("t.pickup_location"),
        F.col("t.drop_location"),
        F.col("t.distance_km"),
        F.col("t.fare_amount"),
        F.col("t.trip_status"),
        F.col("l.log_id"),
        F.col("l.start_time"),
        F.col("l.end_time"),
        F.col("l.delay_minutes"),
        F.col("l.cancellation_flag")
    )
)

print("Joined Silver records:", silver_df.count())
display(silver_df)

Joined Silver records: 150


trip_id,driver_id,driver_name,city,rating,pickup_location,drop_location,distance_km,fare_amount,trip_status,log_id,start_time,end_time,delay_minutes,cancellation_flag
12,62,Anjali_62,Mumbai,4.4,Railway Station,Mall,8.32,0.0,Cancelled,12,2025-01-06T14:05:00.000Z,null,0.0,1
18,57,Priya_57,Pune,4.6,Airport,IT Park,13.23,0.0,Cancelled,18,2025-01-07T15:21:00.000Z,null,0.0,1
38,143,Anjali_143,Mumbai,4.0,Railway Station,IT Park,2.23,32.64,Completed,38,2025-01-04T04:48:00.000Z,2025-01-04T05:39:00.000Z,19.0,0
67,81,Amit_81,Mumbai,4.1,City Center,Airport,7.81,0.0,Cancelled,67,2025-01-04T20:30:00.000Z,null,0.0,1
70,125,Neha_125,Hyderabad,3.5,Railway Station,Mall,4.79,70.95,Completed,70,2025-01-06T08:18:00.000Z,2025-01-06T08:52:00.000Z,9.0,0
93,61,Rahul_61,Bangalore,3.5,Mall,Airport,10.77,0.0,Cancelled,93,2025-01-05T03:33:00.000Z,null,0.0,1
16,150,Sneha_150,Pune,4.8,IT Park,Airport,23.76,0.0,Cancelled,16,2025-01-07T02:27:00.000Z,null,0.0,1
64,150,Sneha_150,Pune,4.8,IT Park,Mall,12.03,164.81,Completed,64,2025-01-04T00:51:00.000Z,2025-01-04T01:24:00.000Z,19.0,0
74,38,Sneha_38,Pune,5.0,Airport,Mall,10.82,156.38,Completed,74,2025-01-02T21:47:00.000Z,2025-01-02T22:47:00.000Z,8.0,0
94,102,Vikas_102,Delhi,4.8,City Center,IT Park,8.31,99.4,Completed,94,2025-01-02T23:52:00.000Z,2025-01-03T00:05:00.000Z,1.0,0


In [0]:
silver_df = (
    silver_df
    .withColumn(
        "trip_duration_minutes",
        F.when(
            F.col("end_time").isNotNull(),
            (F.unix_timestamp("end_time") - F.unix_timestamp("start_time")) / 60
        ).otherwise(None)
    )
    .withColumn(
        "completion_flag",
        F.when(F.col("trip_status") == "Completed", 1).otherwise(0)
    )
    .withColumn(
        "is_delayed",
        F.when(F.col("delay_minutes") > 0, 1).otherwise(0)
    )
    .withColumn(
        "revenue_per_km",
        F.when(
            F.col("distance_km") > 0,
            F.round(F.col("fare_amount") / F.col("distance_km"), 2)
        ).otherwise(0)
    )
)

display(silver_df)

trip_id,driver_id,driver_name,city,rating,pickup_location,drop_location,distance_km,fare_amount,trip_status,log_id,start_time,end_time,delay_minutes,cancellation_flag,trip_duration_minutes,completion_flag,is_delayed,revenue_per_km
12,62,Anjali_62,Mumbai,4.4,Railway Station,Mall,8.32,0.0,Cancelled,12,2025-01-06T14:05:00.000Z,null,0.0,1,null,0,0,0.0
18,57,Priya_57,Pune,4.6,Airport,IT Park,13.23,0.0,Cancelled,18,2025-01-07T15:21:00.000Z,null,0.0,1,null,0,0,0.0
38,143,Anjali_143,Mumbai,4.0,Railway Station,IT Park,2.23,32.64,Completed,38,2025-01-04T04:48:00.000Z,2025-01-04T05:39:00.000Z,19.0,0,51.0,1,1,14.64
67,81,Amit_81,Mumbai,4.1,City Center,Airport,7.81,0.0,Cancelled,67,2025-01-04T20:30:00.000Z,null,0.0,1,null,0,0,0.0
70,125,Neha_125,Hyderabad,3.5,Railway Station,Mall,4.79,70.95,Completed,70,2025-01-06T08:18:00.000Z,2025-01-06T08:52:00.000Z,9.0,0,34.0,1,1,14.81
93,61,Rahul_61,Bangalore,3.5,Mall,Airport,10.77,0.0,Cancelled,93,2025-01-05T03:33:00.000Z,null,0.0,1,null,0,0,0.0
16,150,Sneha_150,Pune,4.8,IT Park,Airport,23.76,0.0,Cancelled,16,2025-01-07T02:27:00.000Z,null,0.0,1,null,0,0,0.0
64,150,Sneha_150,Pune,4.8,IT Park,Mall,12.03,164.81,Completed,64,2025-01-04T00:51:00.000Z,2025-01-04T01:24:00.000Z,19.0,0,33.0,1,1,13.7
74,38,Sneha_38,Pune,5.0,Airport,Mall,10.82,156.38,Completed,74,2025-01-02T21:47:00.000Z,2025-01-02T22:47:00.000Z,8.0,0,60.0,1,1,14.45
94,102,Vikas_102,Delhi,4.8,City Center,IT Park,8.31,99.4,Completed,94,2025-01-02T23:52:00.000Z,2025-01-03T00:05:00.000Z,1.0,0,13.0,1,1,11.96


In [0]:
silver_df = (
    silver_df
    .withColumn(
        "delay_category",
        F.when(F.col("delay_minutes") == 0, "No Delay")
        .when(F.col("delay_minutes") <= 5, "Low Delay")
        .when(F.col("delay_minutes") <= 15, "Medium Delay")
        .otherwise("High Delay")
    )
)

display(
    silver_df.select(
        "trip_id",
        "delay_minutes",
        "delay_category"
    )
)

trip_id,delay_minutes,delay_category
12,0.0,No Delay
18,0.0,No Delay
38,19.0,High Delay
67,0.0,No Delay
70,9.0,Medium Delay
93,0.0,No Delay
16,0.0,No Delay
64,19.0,High Delay
74,8.0,Medium Delay
94,1.0,Low Delay


In [0]:
silver_df = (
    silver_df
    .withColumn("trip_date", F.to_date("start_time"))
    .withColumn("trip_hour", F.hour("start_time"))
)

display(
    silver_df.select(
        "trip_id",
        "start_time",
        "trip_date",
        "trip_hour"
    )
)

trip_id,start_time,trip_date,trip_hour
12,2025-01-06T14:05:00.000Z,2025-01-06,14
18,2025-01-07T15:21:00.000Z,2025-01-07,15
38,2025-01-04T04:48:00.000Z,2025-01-04,4
67,2025-01-04T20:30:00.000Z,2025-01-04,20
70,2025-01-06T08:18:00.000Z,2025-01-06,8
93,2025-01-05T03:33:00.000Z,2025-01-05,3
16,2025-01-07T02:27:00.000Z,2025-01-07,2
64,2025-01-04T00:51:00.000Z,2025-01-04,0
74,2025-01-02T21:47:00.000Z,2025-01-02,21
94,2025-01-02T23:52:00.000Z,2025-01-02,23


In [0]:
silver_output_path = "/Volumes/workspace/default/rideshare_data/silver/ride_data"

(
    silver_df.write
    .mode("overwrite")
    .parquet(silver_output_path)
)

print("Silver layer data saved successfully.")

Silver layer data saved successfully.


In [0]:
silver_check_df = spark.read.parquet(
    "/Volumes/workspace/default/rideshare_data/silver/ride_data"
)

print("Silver record count:", silver_check_df.count())
silver_check_df.printSchema()

Silver record count: 150
root
 |-- trip_id: integer (nullable = true)
 |-- driver_id: integer (nullable = true)
 |-- driver_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- rating: float (nullable = true)
 |-- pickup_location: string (nullable = true)
 |-- drop_location: string (nullable = true)
 |-- distance_km: float (nullable = true)
 |-- fare_amount: float (nullable = true)
 |-- trip_status: string (nullable = true)
 |-- log_id: integer (nullable = true)
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- delay_minutes: float (nullable = true)
 |-- cancellation_flag: integer (nullable = true)
 |-- trip_duration_minutes: double (nullable = true)
 |-- completion_flag: integer (nullable = true)
 |-- is_delayed: integer (nullable = true)
 |-- revenue_per_km: double (nullable = true)
 |-- delay_category: string (nullable = true)
 |-- trip_date: date (nullable = true)
 |-- trip_hour: integer (nullable = true)



In [0]:
silver_summary = [
    ("Source Drivers", drivers_df.count()),
    ("Source Trips", trips_df.count()),
    ("Source Trip Logs", trip_logs_df.count()),
    ("Clean Drivers", clean_drivers_df.count()),
    ("Clean Trips", clean_trips_df.count()),
    ("Clean Trip Logs", clean_trip_logs_df.count()),
    ("Final Silver Records", silver_check_df.count())
]

silver_summary_df = spark.createDataFrame(
    silver_summary,
    ["validation_step", "record_count"]
)

display(silver_summary_df)

validation_step,record_count
Source Drivers,150
Source Trips,150
Source Trip Logs,150
Clean Drivers,150
Clean Trips,150
Clean Trip Logs,150
Final Silver Records,150


## Silver Layer Conclusion

The Silver Layer was completed successfully.

- Bronze datasets were read from Parquet format.
- Duplicate and invalid records were checked.
- Null values were validated using business rules.
- Drivers, trips, and trip logs were joined successfully.
- No records were lost during the join process.
- Derived columns such as trip duration, completion flag, delay category, trip date, and revenue per kilometre were created.
- The final cleaned and enriched dataset contains 150 records.
- The Silver dataset was stored successfully in Parquet format.

The data is now ready for KPI generation, driver performance analysis, revenue analysis